## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [68]:
from fastapi import FastAPI
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langserve import add_routes



load_dotenv(override=True)

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"]="true"

In [69]:
model = ChatGroq(model="openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E0FDD298B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E0FE0E30E0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [70]:
from langchain_core.messages import HumanMessage
result = model.invoke([HumanMessage(content="Hi i am saran, i am an engineer")])
result.content

'Hello Saran! 👋 It’s great to meet an engineer. What field or project are you working on right now? I’m here if you want to bounce ideas, troubleshoot something, or just chat about tech.'

In [71]:
from langchain_core.messages import AIMessage
model.invoke([HumanMessage(content="Hi i am saran, i am an engineer"),
             AIMessage(content='Hello Saran! 👋 It’s great to meet an engineer. What kind of engineering do you specialize in? And how can I assist you today? Whether you’re looking for technical insights, project ideas, coding help, or just a chat about the latest industry trends, I’m here for you!'),
             HumanMessage(content="what is my name and what do i do?")])

AIMessage(content='Your name is **Saran**, and you’re an **engineer**. If you’d like to share more about your field (e.g., civil, electrical, mechanical, software, etc.) or what specific projects you’re working on, I’d love to hear!', additional_kwargs={'reasoning_content': 'User says: "Hi i am saran, i am an engineer". They ask: "what is my name and what do i do?" We must answer: name is Saran, profession engineer. Should we ask for more details? Probably just answer straightforwardly.'}, response_metadata={'token_usage': {'completion_tokens': 119, 'prompt_tokens': 162, 'total_tokens': 281, 'completion_time': 0.134146817, 'completion_tokens_details': {'reasoning_tokens': 55}, 'prompt_time': 0.007766209, 'prompt_tokens_details': None, 'queue_time': 0.284034188, 'total_time': 0.141913026}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d3e146e1a5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019feb68-e38c-

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [72]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

d:\OneDrive - Maarga Systems Private Limited\Documents\Agentic-AI-BridgeCourse\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [73]:
config = {"configurable":{"session_id":"chat1"}}
config

{'configurable': {'session_id': 'chat1'}}

In [74]:
with_message_history.invoke([HumanMessage(content="what is my name?")], config=config)

AIMessage(content='I’m sorry—I don’t have that information. Could you let me know your name?', additional_kwargs={'reasoning_content': 'We need to answer: "what is my name?" The user hasn\'t provided their name. We cannot guess. We should respond politely that we don\'t know. Maybe ask for clarification.'}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 76, 'total_tokens': 140, 'completion_time': 0.067238169, 'completion_tokens_details': {'reasoning_tokens': 37}, 'prompt_time': 0.003644335, 'prompt_tokens_details': None, 'queue_time': 0.280414364, 'total_time': 0.070882504}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e594c51153', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019feb68-e583-7e03-8a4e-ad1297a26c91-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 64, 'total_tokens': 140, 'output_token_details': {'reasoning': 37}})

In [75]:
with_message_history.invoke([HumanMessage(content="my name is saran subramani and people call me saran")], config=config)

AIMessage(content='Nice to meet you, Saran! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says name: Saran Subramani, called Saran. The user is asking "what is my name?" Actually they answered. So we can respond acknowledging. Maybe ask about preferences or ask what they want. Probably just confirm.'}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 118, 'total_tokens': 192, 'completion_time': 0.077336835, 'completion_tokens_details': {'reasoning_tokens': 50}, 'prompt_time': 0.005805552, 'prompt_tokens_details': None, 'queue_time': 0.314185027, 'total_time': 0.083142387}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_21553e1ca5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019feb68-e725-7441-82d8-cb3d3402ec47-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 118, 'output_tokens': 74, 'total_tokens': 192, 'output_token_de

In [76]:
with_message_history.invoke([HumanMessage(content="what is my full name? and how people call me?")], config=config)

AIMessage(content='Your full name is **Saran Subramani**, and people usually call you **Saran**.', additional_kwargs={'reasoning_content': 'The user says "my name is saran subramani and people call me saran". They ask: "what is my full name? and how people call me?" So answer: full name Saran Subramani, people call you Saran. Probably respond accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 155, 'total_tokens': 242, 'completion_time': 0.094052461, 'completion_tokens_details': {'reasoning_tokens': 57}, 'prompt_time': 0.0076457, 'prompt_tokens_details': None, 'queue_time': 0.368114754, 'total_time': 0.101698161}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5979a0e1b7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019feb68-e8fa-7131-8dae-13b5a9c35096-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 155, 'output_tokens': 87, 'total_tokens'

In [77]:
from langchain_core.prompts import MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([("system", "you are a helpful assistant"), 
                                  MessagesPlaceholder(variable_name="messages")])

chain = prompt|model

In [78]:
chain.invoke({"messages":[HumanMessage(content="my name is saran subramani")]})

AIMessage(content='Nice to meet you, Saran! How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says: "my name is saran subramani". They probably want the assistant to respond acknowledging the name or ask a question. They didn\'t ask a question. They just introduced themselves. The instruction: "you are a helpful assistant". So we can respond politely, maybe ask how we can help.'}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 87, 'total_tokens': 175, 'completion_time': 0.092499554, 'completion_tokens_details': {'reasoning_tokens': 64}, 'prompt_time': 0.00418725, 'prompt_tokens_details': None, 'queue_time': 0.284462827, 'total_time': 0.096686804}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_24bfb4a850', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019feb68-eb1b-7fd2-a1e5-2f70382209d4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'in

In [79]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

d:\OneDrive - Maarga Systems Private Limited\Documents\Agentic-AI-BridgeCourse\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [80]:
config = {"configurable":{"session_id":"chat3"}}

In [81]:
response = with_message_history.invoke([HumanMessage(content="my name is saran")],config=config)
response

AIMessage(content='Hello Saran! How can I assist you today?', additional_kwargs={'reasoning_content': 'We just need to respond politely, maybe ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 84, 'total_tokens': 118, 'completion_time': 0.034673817, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.00400785, 'prompt_tokens_details': None, 'queue_time': 0.284663889, 'total_time': 0.038681667}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019feb68-ecec-7341-99c8-5757ec9e7914-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 84, 'output_tokens': 34, 'total_tokens': 118, 'output_token_details': {'reasoning': 14}})

In [82]:
with_message_history.invoke([HumanMessage(content="what is my name?")], config=config)

AIMessage(content='Your name is Saran.', additional_kwargs={'reasoning_content': 'User says name Saran. They ask "what is my name?" So answer Saran.'}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 110, 'total_tokens': 145, 'completion_time': 0.051818417, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.005415896, 'prompt_tokens_details': None, 'queue_time': 0.315116753, 'total_time': 0.057234313}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_21553e1ca5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019feb68-ee69-7633-a545-e6566d8ac788-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 110, 'output_tokens': 35, 'total_tokens': 145, 'output_token_details': {'reasoning': 20}})

In [83]:
prompt = ChatPromptTemplate.from_messages([("system","answer the question in this {language}"),MessagesPlaceholder(variable_name="messages")])

chain = prompt|model

In [84]:
result = chain.invoke({"messages": [HumanMessage(content="hi, my name is saran")], "language":"tamil"})
result

AIMessage(content='வணக்கம், சாரணா! எப்படி இருக்கிறீர்கள்?', additional_kwargs={'reasoning_content': 'The user says: "hi, my name is saran". They want answer in Tamil. The instruction: answer the question in this tamil. There\'s no question. Maybe respond greeting in Tamil. So reply in Tamil: "வணக்கம், சாரணா!" Or "வணக்கம், சாரணா!" Ensure Tamil.'}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 87, 'total_tokens': 177, 'completion_time': 0.098753699, 'completion_tokens_details': {'reasoning_tokens': 67}, 'prompt_time': 0.004168021, 'prompt_tokens_details': None, 'queue_time': 0.247415465, 'total_time': 0.10292172}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e2cb7a84ec', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019feb68-f020-7540-8be8-cb9389103dbf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 90, 'total_tokens': 177, 'outpu

In [85]:
result = prompt.invoke({"messages":[HumanMessage(content="what is my name?")], "language":"tamil"})

print(result.messages)

[SystemMessage(content='answer the question in this tamil', additional_kwargs={}, response_metadata={}), HumanMessage(content='what is my name?', additional_kwargs={}, response_metadata={})]


In [86]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

d:\OneDrive - Maarga Systems Private Limited\Documents\Agentic-AI-BridgeCourse\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [87]:
config = {"configurable": {"session_id":"chat_3"}}
response = with_message_history.invoke({"messages":[HumanMessage(content="i am saran")], "language":"tamil"}, config=config)
response.content

'வணக்கம் சரன்! நான் எப்படி உதவலாம்?'

In [88]:
print(get_session_history("chat_3").messages)

[HumanMessage(content='i am saran', additional_kwargs={}, response_metadata={}), AIMessage(content='வணக்கம் சரன்! நான் எப்படி உதவலாம்?', additional_kwargs={'reasoning_content': 'User says: "i am saran". They likely want to respond in Tamil. The instruction: "answer the question in this tamil". The user didn\'t ask a question; they just say "i am saran". Probably they want a response in Tamil acknowledging them. So respond in Tamil: "வணக்கம் சரன், நான் எப்படி உதவலாம்?" So reply in Tamil.'}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 84, 'total_tokens': 183, 'completion_time': 0.103828005, 'completion_tokens_details': {'reasoning_tokens': 78}, 'prompt_time': 0.003973392, 'prompt_tokens_details': None, 'queue_time': 0.249013785, 'total_time': 0.107801397}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_228717f27c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019feb68-f1d8-75

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [89]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [90]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
         |prompt
         |model
         )

In [91]:
response = chain.invoke({"messages": messages+[HumanMessage(content="what ice do i like?")], "language":"tamil"})
response

AIMessage(content='It sounds like you’re asking about the kind of ice you prefer—whether it’s for drinks, desserts, or something else. Could you tell me a bit more about the context? For example, are you thinking about ice cream flavors, ice for cocktails, or maybe ice used in sports? That’ll help me give you a more tailored answer!', additional_kwargs={'reasoning_content': 'The user asks: "what ice do i like?" They didn\'t specify context. The user previously said "thanks" then "having fun?" then "what ice do i like?" They might be asking about ice flavors? Or "ice" as in "ice cream"? Could be "ice" as in "ice" for sports? Might be "ice" meaning "ice hockey"? They might be asking about preferences. We need to interpret. Possibly they want to know what kind of ice they like: "ice" could be "ice cream" flavors. Or "ice" as in "ice" for a drink? Could be "ice" meaning "ice cream" or "ice" as "ice" for a drink? The user hasn\'t given any context. We can ask clarifying question: "Are you a

In [92]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked for the sum of 2\u202f+\u202f2.'

In [93]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history, input_messages_key="messages")
config = {"configurable": {"session_id":"chat_3"}}
response = with_message_history.invoke({
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config)

response

d:\OneDrive - Maarga Systems Private Limited\Documents\Agentic-AI-BridgeCourse\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AIMessage(content='I’m not sure what your name is—could you let me know?', additional_kwargs={'reasoning_content': 'User asks: "whats my name?" We don\'t know. According to policy: "I do not know the user\'s name unless they tell me." So we should respond that we don\'t know. We can ask them or say we don\'t know. Let\'s comply with policy.'}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 148, 'total_tokens': 227, 'completion_time': 0.086010425, 'completion_tokens_details': {'reasoning_tokens': 55}, 'prompt_time': 0.007179251, 'prompt_tokens_details': None, 'queue_time': 0.284487451, 'total_time': 0.093189676}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d3e146e1a5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019feb68-f865-74e2-a36e-24cf1371681e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 148, 'output_tokens': 79, 'total_tokens': 227, 'outpu

In [94]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'You haven’t asked me a math problem yet! If you have one in mind, feel free to share it and I’ll help you with it.'

In [98]:
print(store["chat1"])

Human: what is my name?
AI: I’m sorry—I don’t have that information. Could you let me know your name?
Human: my name is saran subramani and people call me saran
AI: Nice to meet you, Saran! How can I help you today?
Human: what is my full name? and how people call me?
AI: Your full name is **Saran Subramani**, and people usually call you **Saran**.


In [99]:
print(store["chat3"])

Human: my name is saran
AI: Hello Saran! How can I assist you today?
Human: what is my name?
AI: Your name is Saran.
